# Phase 6 — Test axes other than generic eval loss

Goal: Check whether QGFD's effect shows up on properties the mechanism more plausibly changes, even if generic eval loss stays flat.

Two axes to test:
1. Attention entropy / sink concentration - forward-hook eval examples through both models, compare attention entropy and how much mass sits on position 0 (attention sink).
2. Generation quality - ROUGE-L on held-out Alpagasus prompts against reference outputs.

## 0. Install & Setup

In [ ]:
!pip install -q git+https://github.com/rajboopathiking/TorchDire.git
!pip install -q transformers peft datasets accelerate bitsandbytes scipy matplotlib seaborn pandas scikit-learn evaluate rouge-score

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import torch
import torch.nn.functional as F
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
    TrainingArguments, Trainer, set_seed
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from evaluate import load as load_metric

set_seed(42)

import warnings
warnings.filterwarnings('ignore')

## 1. Configuration

In [ ]:
MODEL_ID = "42dot/42dot_LLM-SFT-1.3B"
DATASET_ID = "arbml/alpagasus_cleaned"

LORA_R = 16
LORA_ALPHA = 32
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

BATCH_SIZE = 2
EVAL_SAMPLES = 100
GEN_SAMPLES = 50
MAX_LENGTH = 512

WORKING_DIR = "/kaggle/working"
os.makedirs(WORKING_DIR, exist_ok=True)

## 2. Load Models (and Data)
Since we might not have checkpoints lying around, let's load the dataset and train brief 300-step models for baseline and QGFD to do the evaluation on.

In [ ]:
def format_prompt(example):
    instruction = example['instruction']
    input_text = example.get('input', '')
    output = example['output']
    
    if input_text:
        prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"
    else:
        prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
        
    return {"prompt": prompt, "full_text": prompt + output}

dataset = load_dataset(DATASET_ID, split="train")
dataset = dataset.map(format_prompt)

eval_dataset = dataset.select(range(EVAL_SAMPLES))
gen_dataset = dataset.select(range(EVAL_SAMPLES, EVAL_SAMPLES + GEN_SAMPLES))
train_dataset = dataset.select(range(EVAL_SAMPLES + GEN_SAMPLES, EVAL_SAMPLES + GEN_SAMPLES + 1500))

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

def tokenize(examples):
    return tokenizer(examples["full_text"], truncation=True, max_length=MAX_LENGTH, padding="max_length")

train_tokenized = train_dataset.map(tokenize, batched=True, remove_columns=train_dataset.column_names)
eval_tokenized = eval_dataset.map(tokenize, batched=True, remove_columns=eval_dataset.column_names)

In [ ]:
from torchdire import register_qgfd_step_callback, patch_llama_with_qgfd, collect_qgfd_kernels

def get_model():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16
    )
    model = prepare_model_for_kbit_training(model)
    
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=LORA_TARGET_MODULES,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )
    
    model = get_peft_model(model, lora_config)
    return model

def train_model(run_name, use_qgfd=False):
    print(f"\n{'='*50}\nTraining {run_name}\n{'='*50}")
    model = get_model()
    
    args = TrainingArguments(
        output_dir=f"{WORKING_DIR}/{run_name}",
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        max_steps=300,
        logging_steps=50,
        save_steps=300,
        bf16=True,
        optim="paged_adamw_32bit",
        report_to="none"
    )
    
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_tokenized,
        data_collator=lambda data: {'input_ids': torch.stack([torch.tensor(f['input_ids']) for f in data]), 
                                    'attention_mask': torch.stack([torch.tensor(f['attention_mask']) for f in data]), 
                                    'labels': torch.stack([torch.tensor(f['input_ids']) for f in data])}
    )
    
    if use_qgfd:
        register_qgfd_step_callback(trainer, target_modules=LORA_TARGET_MODULES, qgfd_start_step=0)
        
    trainer.train()
    model.save_pretrained(f"{WORKING_DIR}/{run_name}_final")
    
    del model
    del trainer
    gc.collect()
    torch.cuda.empty_cache()
    
# Commented out as this will take a while, assuming we run it or load existing.
train_model("baseline", use_qgfd=False)
train_model("qgfd", use_qgfd=True)


## 3. Helper: Attention Hook
Register forward hooks to capture attention weights.

In [ ]:
def get_attention_stats(model_path):
    print(f"\nEvaluating attention for {model_path}")
    model = get_model()
    # Load weights from peft model
    model.load_adapter(f"{WORKING_DIR}/{model_path}_final", "default")
    model.eval()
    
    entropy_stats = []
    sink_stats = []
    
    with torch.no_grad():
        for i, example in enumerate(eval_tokenized):
            if i >= 20: # Limit to 20 samples to save memory and time
                break
                
            input_ids = torch.tensor([example['input_ids']]).to('cuda')
            attention_mask = torch.tensor([example['attention_mask']]).to('cuda')
            
            # Force return attentions
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, output_attentions=True)
            
            attentions = outputs.attentions
            if not attentions:
                print("No attentions returned!")
                break
                
            for layer_idx, layer_attn in enumerate(attentions):
                # layer_attn shape: (batch_size, num_heads, sequence_length, sequence_length)
                # Convert to probability distribution just in case, though they should be softmaxed
                probs = layer_attn.float()
                
                # 4. Attention Entropy Analysis
                # -sum(p * log(p))
                eps = 1e-10
                entropy = -torch.sum(probs * torch.log(probs + eps), dim=-1)
                # Average over sequence length to get per-head entropy
                avg_entropy = entropy.mean(dim=-1)[0] # shape: (num_heads,)
                
                for head_idx, h_ent in enumerate(avg_entropy):
                    entropy_stats.append({
                        'model': model_path,
                        'layer': layer_idx,
                        'head': head_idx,
                        'entropy': h_ent.item()
                    })
                
                # 5. Attention Sink Analysis
                # Mass on position 0
                sink_weights = probs[..., 0] # shape: (batch_size, num_heads, sequence_length)
                avg_sink = sink_weights.mean(dim=-1)[0] # shape: (num_heads,)
                
                for head_idx, h_sink in enumerate(avg_sink):
                    sink_stats.append({
                        'model': model_path,
                        'layer': layer_idx,
                        'head': head_idx,
                        'sink_mass': h_sink.item()
                    })
                    
            del outputs
            del input_ids
            del attention_mask
            torch.cuda.empty_cache()
            
    del model
    gc.collect()
    torch.cuda.empty_cache()
    
    return pd.DataFrame(entropy_stats), pd.DataFrame(sink_stats)

base_ent_df, base_sink_df = get_attention_stats("baseline")
qgfd_ent_df, qgfd_sink_df = get_attention_stats("qgfd")

all_ent_df = pd.concat([base_ent_df, qgfd_ent_df])
all_sink_df = pd.concat([base_sink_df, qgfd_sink_df])

all_ent_df.to_csv(f"{WORKING_DIR}/phase6_entropy.csv", index=False)
all_sink_df.to_csv(f"{WORKING_DIR}/phase6_sink.csv", index=False)

## 6. Entropy Comparison Plots
Violin/box plots of attention entropy distribution, layer-wise comparison.

In [ ]:
plt.figure(figsize=(15, 6))
sns.violinplot(data=all_ent_df, x='layer', y='entropy', hue='model', split=True)
plt.title('Attention Entropy per Layer (Baseline vs QGFD)')
plt.xlabel('Layer')
plt.ylabel('Entropy')
plt.savefig(f"{WORKING_DIR}/phase6_entropy_plot.png")
plt.show()

plt.figure(figsize=(15, 6))
sns.violinplot(data=all_sink_df, x='layer', y='sink_mass', hue='model', split=True)
plt.title('Attention Sink Mass (Position 0) per Layer (Baseline vs QGFD)')
plt.xlabel('Layer')
plt.ylabel('Sink Mass Probability')
plt.savefig(f"{WORKING_DIR}/phase6_sink_plot.png")
plt.show()

## 7. Generation Quality
Generate responses on 50 held-out Alpagasus prompts with both models, compute ROUGE-L.

In [ ]:
rouge = load_metric("rouge")

def generate_responses(model_path):
    print(f"\nGenerating for {model_path}")
    model = get_model()
    model.load_adapter(f"{WORKING_DIR}/{model_path}_final", "default")
    model.eval()
    
    responses = []
    references = []
    
    tokenizer.padding_side = 'left' # For generation
    
    with torch.no_grad():
        for idx in range(len(gen_dataset)):
            prompt = gen_dataset[idx]['prompt']
            ref = gen_dataset[idx]['output']
            
            inputs = tokenizer(prompt, return_tensors="pt").to('cuda')
            
            outputs = model.generate(
                **inputs,
                max_new_tokens=128,
                temperature=0.7,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )
            
            gen_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
            
            responses.append(gen_text)
            references.append(ref)
            
            if (idx + 1) % 10 == 0:
                print(f"Generated {idx+1}/{len(gen_dataset)}")
                
    del model
    gc.collect()
    torch.cuda.empty_cache()
    
    results = rouge.compute(predictions=responses, references=references)
    return results, responses, references

base_rouge, base_res, refs = generate_responses("baseline")
qgfd_rouge, qgfd_res, _ = generate_responses("qgfd")

## 8. ROUGE Results Table & Plot

In [ ]:
metrics_df = pd.DataFrame([
    {'Model': 'Baseline', 'ROUGE-1': base_rouge['rouge1'], 'ROUGE-2': base_rouge['rouge2'], 'ROUGE-L': base_rouge['rougeL']},
    {'Model': 'QGFD', 'ROUGE-1': qgfd_rouge['rouge1'], 'ROUGE-2': qgfd_rouge['rouge2'], 'ROUGE-L': qgfd_rouge['rougeL']}
])

metrics_df.to_csv(f"{WORKING_DIR}/phase6_rouge.csv", index=False)
print(metrics_df)

metrics_df.set_index('Model').plot(kind='bar', figsize=(10, 6))
plt.title('Generation Quality (ROUGE Metrics)')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.savefig(f"{WORKING_DIR}/phase6_rouge_plot.png")
plt.show()

## 9. Combined Analysis
Do any of these alternative axes show a QGFD effect?
- Check if attention entropy differs, meaning QGFD causes sharper/flatter attention.
- Check if sink concentration differs, meaning QGFD impacts the attention sink allocation.
- Check if generation quality (ROUGE-L) actually improves even if eval loss was flat.

## 10. Final Summary
- Attention Entropy insights:
- Attention Sink insights:
- Generation Quality insights:
Overall conclusion on whether QGFD modifies internal representations or generation behavior meaningfully.